In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.inspection import permutation_importance

X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

preprocessor = joblib.load("../models/preprocessor.pkl")
best_ridge = joblib.load("../models/ridge_tuned.pkl")

X_train_processed = preprocessor.transform(X_train)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [2]:
feature_names = preprocessor.get_feature_names_out()
print(len(feature_names))
print(feature_names[:10])

311
['num__MSSubClass' 'num__LotFrontage' 'num__LotArea' 'num__OverallQual'
 'num__OverallCond' 'num__YearBuilt' 'num__YearRemodAdd' 'num__MasVnrArea'
 'num__BsmtFinSF1' 'num__BsmtFinSF2']


In [3]:
best_ridge.fit(X_train_processed, y_train)

perm_result = permutation_importance(
    best_ridge, X_train_processed, y_train,
    n_repeats=10, random_state=42, scoring='neg_root_mean_squared_error', n_jobs=-1
)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std
}).sort_values('importance_mean', ascending=False)

print(importance_df.head(20))
print("\nBottom 20 (least useful / possibly harmful):")
print(importance_df.tail(20))

                       feature  importance_mean  importance_std
15              num__GrLivArea         0.022496        0.001373
36                num__TotalSF         0.018376        0.001152
42      num__TotalQualityScore         0.012715        0.000770
3             num__OverallQual         0.012006        0.000900
13               num__2ndFlrSF         0.009640        0.000622
12               num__1stFlrSF         0.008616        0.000826
32               num__PoolArea         0.007172        0.000646
38               num__HouseAge         0.007136        0.000778
5               num__YearBuilt         0.007085        0.000771
44                num__HasPool         0.005980        0.000628
8              num__BsmtFinSF1         0.005726        0.000767
11            num__TotalBsmtSF         0.005336        0.000559
4             num__OverallCond         0.003554        0.000352
2                 num__LotArea         0.003495        0.000437
25             num__GarageCars         0

In [4]:
# Check correlation among top numeric engineered + original features
check_cols = ['TotalSF', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
              'GarageCars', 'GarageArea', 'TotalQualityScore', 'OverallQual', 'OverallCond']

corr_matrix = X_train[check_cols].corr()
print(corr_matrix.round(2))

                   TotalSF  GrLivArea  TotalBsmtSF  1stFlrSF  2ndFlrSF  \
TotalSF               1.00       0.87         0.81      0.77      0.35   
GrLivArea             0.87       1.00         0.41      0.53      0.70   
TotalBsmtSF           0.81       0.41         1.00      0.80     -0.19   
1stFlrSF              0.77       0.53         0.80      1.00     -0.23   
2ndFlrSF              0.35       0.70        -0.19     -0.23      1.00   
GarageCars            0.56       0.49         0.45      0.45      0.19   
GarageArea            0.56       0.46         0.48      0.48      0.14   
TotalQualityScore     0.43       0.43         0.29      0.26      0.27   
OverallQual           0.68       0.59         0.54      0.46      0.29   
OverallCond          -0.15      -0.08        -0.18     -0.15      0.03   

                   GarageCars  GarageArea  TotalQualityScore  OverallQual  \
TotalSF                  0.56        0.56               0.43         0.68   
GrLivArea                0.49  

In [5]:
low_importance_features = importance_df[importance_df['importance_mean'] <= 0]['feature'].tolist()
print(f"Number of low/negative importance features: {len(low_importance_features)}")
print(low_importance_features)

Number of low/negative importance features: 11
['cat__Condition2_Feedr', 'num__GarageAge', 'cat__Fence_GdPrv', 'cat__RoofStyle_Flat', 'cat__Condition1_RRNn', 'cat__Functional_Min2', 'cat__FireplaceQu_Ex', 'cat__Exterior2nd_VinylSd', 'cat__HeatingQC_Fa', 'cat__MiscFeature_Shed', 'cat__BsmtExposure_Missing']


In [6]:
X_train_reduced = X_train.drop(columns=['TotalQualityScore', 'HasPool'])

# Rebuild a fresh preprocessor for the reduced feature set
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_cols_r = X_train_reduced.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols_r = X_train_reduced.select_dtypes(include=['object']).columns.tolist()

numerical_pipeline_r = Pipeline([('imputer', SimpleImputer(strategy='median')),
                                   ('scaler', StandardScaler())])
categorical_pipeline_r = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
                                     ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor_reduced = ColumnTransformer([
    ('num', numerical_pipeline_r, numerical_cols_r),
    ('cat', categorical_pipeline_r, categorical_cols_r)
])

X_train_reduced_processed = preprocessor_reduced.fit_transform(X_train_reduced)

/var/folders/xd/r1tqvb4s2bvd6gpjtvvbp53r0000gn/T/ipykernel_4594/184306761.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_r = X_train_reduced.select_dtypes(include=['object']).columns.tolist()


In [10]:
best_ridge_loaded = joblib.load("../models/ridge_tuned.pkl")
best_alpha = best_ridge_loaded.alpha
print("Best alpha:", best_alpha)

Best alpha: 20.0


In [11]:
ridge_full = Ridge(alpha=best_alpha, random_state=42)
score_full = -cross_val_score(ridge_full, X_train_processed, y_train,
                                cv=cv, scoring='neg_root_mean_squared_error').mean()

ridge_reduced = Ridge(alpha=best_alpha, random_state=42)
score_reduced = -cross_val_score(ridge_reduced, X_train_reduced_processed, y_train,
                                   cv=cv, scoring='neg_root_mean_squared_error').mean()

print(f"Full feature set RMSE: {score_full:.4f}")
print(f"Reduced feature set RMSE: {score_reduced:.4f}")

Full feature set RMSE: 0.1140
Reduced feature set RMSE: 0.1147
